# Volume 10, Chapter 1: Independence and zero–one laws (Python companion)

Exercises first, solutions at the end. We watch zero–one laws decide random power series and random walks, compute the density of the random harmonic series $\sum\pm\frac1n$ to many digits (and its near miss at $x=2$), test the Paley–Zygmund bound behind the Rademacher–Khintchine–Kolmogorov theorem, and finish with a computer-science application of random signs: **balancing vectors**, the first step of discrepancy theory and of randomized rounding. Requires `numpy` and `scipy`.

In [ ]:
import math
import numpy as np
from scipy.integrate import quad
rng = np.random.default_rng(10)

## Exercises

**P1.1 (★) Random power series.** For i.i.d. coefficients $X_n$, the radius of convergence of $\sum X_nz^n$ is $1/\limsup|X_n|^{1/n}$, almost surely constant (Exercise 1.5): $1$ if $\mathbb E\log^+|X|<\infty$, $0$ otherwise. Estimate $\max_{N/2<n\le N}|X_n|^{1/n}$ for $N=10^3,10^4,10^5,10^6$ and three laws: standard Gaussian, standard Cauchy, and $X=e^{|C|}$ with $C$ Cauchy (for which $\mathbb E\log^+X=\infty$). Run several independent samples: do they agree, as the zero–one law demands?

**P1.2 (★) The fate of a random walk.** Simulate walks with Gaussian steps of mean $\mu\in\{0.1,0,-0.1\}$ and variance $1$, for $10^6$ steps. Count the sign changes of $S_n$ and record $\max_{n\le N}S_n/\sqrt n$ for $N=10^2,\dots,10^6$. Which quantity grows for the centred walk (Corollary 1.9), and which stays bounded for the others (Theorem 1.8)?

**P1.3 (★★) The random harmonic series.** Compute the density $g$ of $X=\sum\varepsilon_n/n$ from $g(x)=\frac1\pi\int_0^\infty\cos(xt)\prod_n\cos\frac tn\dd t$ (Proposition 1.16). Evaluate $g(0)$ and $g(2)$, compare with a histogram of $10^6$ simulated sums (truncated at $n=2000$, plus a Gaussian for the remainder), and compute $P(X\ge4)$ from $P(X>x)=\frac12-\frac1\pi\int_0^\infty\frac{\sin xt}t\varphi(t)\dd t$. Compare with the bound $e^{-3x^2/\pi^2}$ of Exercise 1.2.

**P1.4 (★★) Paley–Zygmund for random signs.** For $a=(1,\dots,1)$ (a simple random walk) and $a_k=1/\sqrt k$, compute exactly or by simulation $\mathbb E T^4/\sigma^4$ and $P(T^2>\sigma^2/2)$, and compare with the bounds $3$ and $\frac1{12}$ of Lemma 1.14. Then simulate $S_n=\sum_{k\le n}\varepsilon_k/\sqrt k$ up to $n=10^6$ (here $\sum a_k^2=\infty$): plot or tabulate $\max_{k\le n}S_k$ and $\min_{k\le n}S_k$ against $\sqrt{\log n}$ (Exercise 1.7 says both are unbounded).

**P1.5 (★★★) CS application: balancing vectors with random signs.** Given $n$ vectors $v_1,\dots,v_n\in[-1,1]^m$, we want signs $\varepsilon_i$ making $\|\sum\varepsilon_iv_i\|_\infty$ small (splitting a set of jobs into two halves balanced on $m$ resources at once, or rounding a fractional solution). (a) Show with Hoeffding's inequality and a union bound that random signs achieve $\|\sum\varepsilon_iv_i\|_\infty\le\sqrt{2n\log(4m)}$ with probability at least $\frac12$. (b) Implement random signing (best of 20 tries) and a greedy method (choose each sign to minimise the current $\infty$-norm) for $n=m=400$ random $\pm1$ vectors, and compare with $\sqrt{2n\log(4m)}$. (Spencer's theorem says $6\sqrt n$ is always possible when $m=n$: randomness alone loses a $\sqrt{\log n}$ factor, and recovering it is the subject of discrepancy theory.)

## Solutions

### P1.1
**How to find it.** Draw the samples in blocks and take the maximum of $\log|X_n|/n$; for $e^{|C|}$, $\log X=|C|$ directly, which avoids overflow.

In [ ]:
def max_root(logabs_sampler, N):
    n = np.arange(N // 2 + 1, N + 1)
    return float(np.exp(np.max(logabs_sampler(len(n)) / n)))

laws = {"Gaussian": lambda k: np.log(np.abs(rng.standard_normal(k))),
        "Cauchy": lambda k: np.log(np.abs(rng.standard_cauchy(k))),
        "exp|Cauchy|": lambda k: np.abs(rng.standard_cauchy(k))}
for name, s in laws.items():
    rows = [[round(max_root(s, N), 4) for N in (10**3, 10**4, 10**5, 10**6)] for _ in range(3)]
    print(f"{name:12s}", rows)

For the Gaussian and the Cauchy laws the maxima approach $1$ in every run (the Cauchy law has $\mathbb E\log^+|X|<\infty$ although it has no mean), so $R=1$. For $e^{|C|}$ they are large and erratic but grow: $\log X=|C|$ exceeds $cn$ for infinitely many $n$, whatever $c$, so $R=0$. Different runs agree on the limit, not on the finite values.

### P1.2
**How to find it.** `np.cumsum` of the steps; a sign change is an index where `S[n-1]*S[n] < 0`.

In [ ]:
N = 10**6
for mu in (0.1, 0.0, -0.1):
    S = np.cumsum(rng.normal(mu, 1.0, N))
    changes = int(np.sum(S[:-1] * S[1:] < 0))
    n = np.arange(1, N + 1)
    ratios = [round(float(np.max(S[:M] / np.sqrt(n[:M]))), 2) for M in (10**2, 10**3, 10**4, 10**5, 10**6)]
    print(f"mu={mu:+.1f}: sign changes {changes:6d}; max S_n/sqrt(n) up to 1e2..1e6: {ratios}")

With a drift the walk changes sign only a few times before leaving for good, and $\max S_n/\sqrt n$ either grows like $\mu\sqrt N$ (for $\mu>0$) or stabilises (for $\mu<0$). For the centred walk the sign changes keep coming (their number grows like $\sqrt N$, with wide fluctuations from run to run) and $\max_{n\le N}S_n/\sqrt n$ creeps up, slowly: the law of the iterated logarithm (Chapter 3) says like $\sqrt{2\log\log N}$.

### P1.3
**How to find it.** Multiply the first $2000$ cosines directly and sum the logarithms of the next ones (all positive once $t/n<\pi/2$); $|\varphi(t)|\le e^{-\pi t/6}$, so integrating up to $t=120$ leaves an error below $10^{-27}$. The limit of double precision is the quadrature, about $10^{-13}$.

In [ ]:
nn = np.arange(1, 10**5 + 1, dtype=float)
def phi(t):
    x = t / nn
    return np.prod(np.cos(x[:2000])) * np.exp(np.sum(np.log(np.abs(np.cos(x[2000:])))))
def g(x):
    return quad(lambda t: math.cos(x * t) * phi(t), 0, 120, limit=1000)[0] / math.pi
def tail(x):
    return 0.5 - quad(lambda t: math.sin(x * t) / t * phi(t), 0, 120, limit=2000)[0] / math.pi

print("g(0) =", g(0), "   g(0) - 1/4 =", g(0) - 0.25)
print("g(2) =", g(2), "   g(2) - 1/8 =", g(2) - 0.125, "(double precision noise; Schmuland: below 1e-42)")

# simulation: first 2000 terms exactly, remainder as a Gaussian of variance sum_{n>2000} 1/n^2
M, K = 2000, 2 * 10**5
X = np.zeros(K)
for start in range(1, M + 1, 500):
    block = np.arange(start, min(start + 500, M + 1))
    X += (rng.integers(0, 2, (K, len(block))) * 2 - 1) @ (1.0 / block)
X += rng.normal(0, math.sqrt(math.pi**2 / 6 - np.sum(1.0 / np.arange(1, M + 1)**2)), K)
for x0 in (0.0, 1.0, 2.0):
    h = 0.05
    print(f"histogram density near {x0}: {np.mean(np.abs(X - x0) < h) / (2 * h):.4f}   g = {g(x0):.4f}")
for x in (3, 4):
    print(f"P(X>={x}) = {tail(x):.3e}   Chernoff bound exp(-3x^2/pi^2) = {math.exp(-3 * x * x / math.pi**2):.3e}")

The histogram agrees with $g$ to within the sampling error, about $0.005$ here. The value $g(2)$ agrees with $\frac18$ to the precision of the computation; Schmuland found it agrees to 42 digits but not exactly. The tail is far thinner than the sub-Gaussian bound: $P(X\ge4)\approx2\times10^{-7}$, against $7.7\times10^{-3}$.

### P1.4
**How to find it.** For $a=(1,\dots,1)$, $T=S_n$ is a simple random walk and $\mathbb E S_n^4=3n^2-2n$, so the ratio tends to $3$: the bound of Lemma 1.14 is sharp. The probability $P(T^2>\sigma^2/2)$ tends to $P(Z^2>\frac12)\approx0.48$, far above $\frac1{12}$, which is the price of a bound that holds for all coefficient vectors.

In [ ]:
from math import comb
for n in (10, 100, 1000):
    # exact law of a simple random walk: S = 2k - n with probability C(n,k)/2^n
    ks = np.arange(n + 1); pk = np.array([comb(n, int(k)) for k in ks], dtype=float) / 2.0**n; S = 2 * ks - n
    m4 = float(np.sum(pk * S**4.0)); pz = float(np.sum(pk[S**2 > n / 2]))
    print(f"a=1, n={n}: E T^4/sigma^4 = {m4 / n**2:.4f} (bound 3),  P(T^2 > sigma^2/2) = {pz:.4f} (bound 1/12)")

n = 10**6
S = np.cumsum((rng.integers(0, 2, n) * 2 - 1) / np.sqrt(np.arange(1, n + 1)))
for m in (10**2, 10**3, 10**4, 10**5, 10**6):
    print(f"n={m:8d}: max S = {S[:m].max():6.3f}, min S = {S[:m].min():6.3f}, sqrt(log n) = {math.sqrt(math.log(m)):.3f}")

Here $\sigma_n^2=\sum_{k\le n}1/k\approx\log n$, so $S_n$ is of order $\sqrt{\log n}$: the walk is unbounded, as Exercise 1.7 proves, but it spreads at the slowest pace one can imagine. A single run up to $10^6$ reaches only about $\pm3$.

### P1.5
**How to find it.** (a) Each coordinate $\sum_i\varepsilon_iv_{ij}$ is a sum of independent bounded centred terms, $|\varepsilon_iv_{ij}|\le1$, so Hoeffding (Volume 3, Theorem 4.9) gives $P(|\sum_i\varepsilon_iv_{ij}|\ge\lambda)\le2e^{-\lambda^2/2n}$. A union bound over the $m$ coordinates gives failure probability at most $2me^{-\lambda^2/2n}$, which is $\frac12$ for $\lambda=\sqrt{2n\log(4m)}$. (b) The greedy method keeps the running sum and picks the sign that gives the smaller $\infty$-norm.

In [ ]:
n = m = 400
V = rng.choice([-1.0, 1.0], size=(n, m))
bound = math.sqrt(2 * n * math.log(4 * m))
best_random = min(np.abs(rng.choice([-1.0, 1.0], n) @ V).max() for _ in range(20))
s = np.zeros(m)
for v in V:
    s = s + v if np.abs(s + v).max() <= np.abs(s - v).max() else s - v
print(f"Hoeffding + union bound: {bound:.1f};  random signs (best of 20): {best_random:.0f};  greedy: {np.abs(s).max():.0f};  6 sqrt(n) = {6 * math.sqrt(n):.0f}")

Random signs land below the Hoeffding–union bound, as expected, and the greedy method does better still on these random inputs; both are even below Spencer's worst-case guarantee $6\sqrt n$. The difference shows on adversarial inputs: there, random signs cannot beat order $\sqrt{n\log n}$, while Spencer's theorem (1985) guarantees $6\sqrt n$. His proof was not an algorithm, and finding such signs efficiently remained open until Bansal's algorithm of 2010.